Import relevant libraries

In [ ]:
import numpy as np
from scipy.optimize import least_squares

**Problem 3e**

Define a function that returns the excess demand (i.e., total consumption for each time and state minus the total endowments)

In [2]:
def excess_demand(x: np.array) -> np.array:
    '''
    Args: 
    - x (np.array): vector of length 2 containing the initial guess for the log of the state prices psi1, psi2
    
    Returns:
    - An array of length 3 containing the excess demand for each time and state
    '''

    psi1, psi2 = np.exp(x) # Enforce positivity of state prices

    # Consumption demand for agent 1
    c1_0 = 0.5*(1 + 2*psi1 + 5*psi2) # Consumption demand for agent 1 at time 0
    c1_1_1 = (1/(2*psi1)) * c1_0 # Consumption demand for agent 1 at time 1, state 1
    c1_1_2 = (1/(2*psi2)) * c1_0 # Consumption demand for agent 1 at time 1, state 2

    # Consumption demand for agent 2
    c2_0 = (1 + 4*psi1 + 3*psi2) * (4*psi1*psi2)/(4*psi1*psi2 + psi2 + psi1) # Consumption demand for agent 2 at time 0
    c2_1_1 = c2_0 / (4*(psi1**2)) # Consumption demand for agent 2 at time 1, state 1
    c2_1_2 = c2_0 / (4*(psi2**2)) # Consumption demand for agent 2 at time 1, state 2

    # Excess demand (derived from market clearing conditions)
    z_0 = c1_0 + c2_0 - 2 # Time 0
    z_1_1 = c1_1_1 + c2_1_1 - 6 # Time 1, state 1
    z_1_2 = c1_1_2 + c2_1_2 - 8 # Time 1, state 2

    return np.array([z_0, z_1_1, z_1_2]) 

Find the state prices (the values of psi1, psi2 that equalize the excess demand to 0) using least_squares from scipy optimize

In [3]:
# Initial guess for the state prices
# Note: We consider an initial guess for the state prices that:
# 1) Is symmetric, and 
# 2) Takes values between 0 and 1 (reasonable for prices of AD securities)
psi1_0 = 0.2
psi2_0 = 0.2

# Solve the system of equations
sol = least_squares(excess_demand, x0=np.log([psi1_0, psi2_0])) 
sol

     message: `gtol` termination condition is satisfied.
     success: True
      status: 1
         fun: [-1.332e-15  3.553e-15  3.553e-15]
           x: [-1.438e+00 -1.619e+00]
        cost: 1.3509243001909827e-29
         jac: [[ 7.676e-01  9.666e-01]
               [-6.552e+00  3.134e+00]
               [ 3.977e+00 -8.636e+00]]
        grad: [-1.017e-14 -2.083e-14]
  optimality: 2.0833152574453003e-14
 active_mask: [ 0.000e+00  0.000e+00]
        nfev: 5
        njev: 5

The solution is expressed in terms of the log of the state prices. So, take the exponentials

In [4]:
psi1, psi2 = np.exp(sol.x)
psi1, psi2

(np.float64(0.23740104123287764), np.float64(0.19809400988878864))

**Problem 3f**

In [5]:
# Consumption demand for agent 1
c1_0 = 0.5*(1 + 2*psi1 + 5*psi2) # Consumption demand for agent 1 at time 0
c1_1_1 = (1/(2*psi1)) * c1_0 # Consumption demand for agent 1 at time 1, state 1
c1_1_2 = (1/(2*psi2)) * c1_0 # Consumption demand for agent 1 at time 1, state 2
print(f'''Allocation for agent 1:
Time 0: {c1_0}
Time 1, state 1: {c1_1_1}
Time 1, state 2: {c1_1_2}
''')

# Consumption demand for agent 2
c2_0 = (1 + 4*psi1 + 3*psi2) * (4*psi1*psi2)/(4*psi1*psi2 + psi2 + psi1) # Consumption demand for agent 2 at time 0
c2_1_1 = c2_0 / (4*(psi1**2)) # Consumption demand for agent 2 at time 1, state 1
c2_1_2 = c2_0 / (4*(psi2**2)) # Consumption demand for agent 2 at time 1, state 2
print(f'''Allocation for agent 2:
Time 0: {c2_0}
Time 1, state 1: {c2_1_1}
Time 1, state 2: {c2_1_2}
''')


Allocation for agent 1:
Time 0: 1.2326360659548492
Time 1, state 1: 2.5961050119104145
Time 1, state 2: 3.1112401294891745

Allocation for agent 2:
Time 0: 0.7673639340451496
Time 1, state 1: 3.403894988089589
Time 1, state 2: 4.88875987051083



In [6]:
c1_1_1/c1_1_2

np.float64(0.8344277213783114)

In [7]:
c2_1_1/c2_1_2

np.float64(0.6962696222046009)

**Problem 3g**

In [8]:
r_f = 1/(psi1+psi2)
print(f'R_f = {r_f:.3f}')

R_f = 2.296


**Problem 3j**

Define a function that returns the system of equations that we want to solve

In [9]:
def system(x: np.array) -> np.array:
    '''
    Args: 
    - x (np.array): vector of length 2 containing the initial guess for the log of the welfare weights (lambda1, lambda2)
    
    Returns:
    - An array of length 2 containing the system of equations we want to solve
    '''

    lambda1, lambda2 = np.exp(x) # Enforce positivity
    # System of equations
    eq1 = (1/192)*(8*lambda1 + np.sqrt((8*lambda1)**2 + 384*(lambda2)**2)) - psi1
    eq2 = (1/256)*(8*lambda1 + np.sqrt((8*lambda1)**2 + 512*(lambda2)**2)) - psi2

    return np.array([eq1, eq2])

Find the optimal welfare weights using least_squares from scipy optimize

In [10]:
# Initial guess for the welfare weights
lambda1_0 = 0.5
lambda2_0 = 0.5

# Solve the system of equations
sol = least_squares(system, x0=np.log([lambda1_0, lambda2_0])) 
sol

     message: `gtol` termination condition is satisfied.
     success: True
      status: 1
         fun: [ 1.188e-11  9.511e-12]
           x: [ 2.092e-01  5.608e-01]
        cost: 1.158549323132783e-22
         jac: [[ 6.554e-02  1.719e-01]
               [ 4.782e-02  1.503e-01]]
        grad: [ 1.234e-12  3.472e-12]
  optimality: 3.4718428514809097e-12
 active_mask: [ 0.000e+00  0.000e+00]
        nfev: 6
        njev: 6

The solution is expressed in terms of the log of the welfare weights. So, take the exponentials

In [11]:
lambda1, lambda2 = np.exp(sol.x)
print(f'''Welfare weights:
lambda1 = {lambda1}
lambda2 = {lambda2}
''')

Welfare weights:
lambda1 = 1.2326360660692603
lambda2 = 1.7519862260839338



In [12]:
c1_0 = lambda1
c1_1_1 = lambda1/(2*psi1)
c1_1_2 = lambda1/(2*psi2)
c2_0 = 0.25*(lambda2**2)
c2_1_1 = 0.0625*(lambda2**2)/(psi1**2)
c2_1_2 = 0.0625*(lambda2**2)/(psi2**2)

print(f'''Allocation for agent 1:
Time 0: {c1_0}
Time 1, state 1: {c1_1_1}
Time 1, state 2: {c1_1_2}
''')
print(f'''Allocation for agent 2:
Time 0: {c2_0}
Time 1, state 1: {c2_1_1}
Time 1, state 2: {c2_1_2}
''')

Allocation for agent 1:
Time 0: 1.2326360660692603
Time 1, state 1: 2.5961050121513805
Time 1, state 2: 3.1112401297779546

Allocation for agent 2:
Time 0: 0.7673639340969562
Time 1, state 1: 3.4038949883193945
Time 1, state 2: 4.888759870840882



**Problem 4d**

Define a function that returns the excess demand (i.e., total consumption for each time and state minus the total endowments)

In [13]:
def excess_demand_4d(x: np.array) -> np.array:
    '''
    Args: 
    - x (np.array): vector of length 2 containing the initial guess for the security prices p1, p2
    
    Returns:
    - An array of length 3 containing the excess demand for each time and state
    '''

    p1, p2 = x

    # Consumption demand for agent 1
    c1_0 = (10/19)*(1+0.75*p1+0.5*p2) # Consumption demand for agent 1 at time 0
    c1_1_1 = ((9/20)*c1_0)/(0.75*p1-0.5*p2) # Consumption demand for agent 1 at time 1, state 1  
    c1_1_2 = ((9/20)*c1_0)/(-0.25*p1+0.5*p2) # Consumption demand for agent 1 at time 1, state 2

    # Consumption demand for agent 2
    c2_0 = 0.5 + (7/8)*p1 - 0.25*p2 # Consumption demand for agent 2 at time 0
    c2_1_1 = c2_0/(1.5*p1-p2) # Consumption demand for agent 2 at time 1, state 1
    c2_1_2 = c2_0/(-0.5*p1+p2) # Consumption demand for agent 2 at time 1, state 2

    # Excess demand (derived from market clearing conditions)
    z_0 = c1_0 + c2_0 - 2 # Time 0
    z_1_1 = c1_1_1 + c2_1_1 - 5 # Time 1, state 1
    z_1_2 = c1_1_2 + c2_1_2 - 5 # Time 1, state 2

    return np.array([z_0, z_1_1, z_1_2]) 

Find the equilibrium security prices using least_squares from scipy optimize

In [14]:
# Initial guess for the security prices
p1_0 = 1
p2_0 = 1

# Solve the system of equations
sol = least_squares(excess_demand_4d, x0=np.array([p1_0, p2_0])) 
p1, p2 = sol.x
p1, p2

(np.float64(0.7589743589743593), np.float64(0.7589743589743593))

**Problem 4e**

In [15]:
D = np.array([[2, 2], [1, 3]])
psi1, psi2 = np.linalg.solve(D, np.array([p1, p2]))
psi1, psi2

(np.float64(0.18974358974358982), np.float64(0.18974358974358982))

**Problem 4f**

In [16]:
# Consumption demand for agent 1
c1_0 = (10/19)*(1+0.75*p1+0.5*p2) # Consumption demand for agent 1 at time 0
c1_1_1 = ((9/20)*c1_0)/(0.75*p1-0.5*p2) # Consumption demand for agent 1 at time 1, state 1  
c1_1_2 = ((9/20)*c1_0)/(-0.25*p1+0.5*p2) # Consumption demand for agent 1 at time 1, state 2

# Consumption demand for agent 2
c2_0 = 0.5 + (7/8)*p1 - 0.25*p2 # Consumption demand for agent 2 at time 0
c2_1_1 = c2_0/(1.5*p1-p2) # Consumption demand for agent 2 at time 1, state 1
c2_1_2 = c2_0/(-0.5*p1+p2) # Consumption demand for agent 2 at time 1, state 2

print(f'''Allocation for agent 1:
Time 0: {c1_0}
Time 1, state 1: {c1_1_1}
Time 1, state 2: {c1_1_2}
''')
print(f'''Allocation for agent 2:
Time 0: {c2_0}
Time 1, state 1: {c2_1_1}
Time 1, state 2: {c2_1_2}
''')

Allocation for agent 1:
Time 0: 1.0256410256410258
Time 1, state 1: 2.4324324324324325
Time 1, state 2: 2.432432432432432

Allocation for agent 2:
Time 0: 0.9743589743589746
Time 1, state 1: 2.5675675675675675
Time 1, state 2: 2.567567567567567



In [17]:
# Security holdings for agent 1
theta1_1 = 0.75*c1_1_1 - 0.25*c1_1_2 - 0.75 # Agent 1's holdings of security 1
theta1_2 = -0.5*c1_1_1 + 0.5*c1_1_2 - 0.5 # Agent 1's holdings of security 2

# Security holdings for agent 2
theta2_1 = 0.75*c2_1_1 - 0.25*c2_1_2 - (7/4) # Agent 2's holdings of security 1
theta2_2 = -0.5*c2_1_1 + 0.5*c2_1_2 + 0.5 # Agent 2's holdings of security 2

print(f'''Security holdings for agent 1:
Security 1: {theta1_1}
Security 2: {theta1_2}
''')
print(f'''Security holdings for agent 2:
Security 1: {theta2_1}
Security 2: {theta2_2}
''')


Security holdings for agent 1:
Security 1: 0.46621621621621623
Security 2: -0.5000000000000002

Security holdings for agent 2:
Security 1: -0.46621621621621623
Security 2: 0.4999999999999998



**Problem 4g**

In [18]:
r_f = 1/(psi1+psi2)
print(f'R_f = {r_f:.3f}')

R_f = 2.635
